***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [8.2 第一代校准（1GC）](8_2_1gc.ipynb)
    * 下一节： [8.4 第三代校准（3GC）](8_4_3gc.ipynb)

***


## 8.3 第二代校准（2GC）：方向无关自校准

第二代校准（2GC）通常指方向无关自校准。与 1GC 依赖外场校准源不同，自校准直接利用目标场数据本身求解增益，再把改正后的数据送回成像和去卷积，形成“成像、建模、求解、改正”的闭环。它能够显著提高动态范围，是现代射电成像中最有力的技术之一。

自校准的风险也来自同一个闭环。求解器并不知道天空模型是否完整，只会根据当前模型最小化残差。若模型漏掉结构、解间隔选择不当，或者方向依赖误差被强行塞进方向无关增益，图像可以变得更平滑，却未必更接近真实天空。因此，自校准必须被理解成模型驱动的反问题，而不是自动改善图像的魔法步骤。


### 8.3.1 自校准闭环

自校准的基本循环可以写成：先用当前数据成像并去卷积，得到一个天空模型；再用该模型预测可见度并求解天线增益；随后应用增益改正数据，回到下一轮成像。若每轮模型都更接近真实天空，增益解会逐步从天空结构中分离出来，图像动态范围也随之提高。

这个过程隐含一个重要条件：当前天空模型必须已经包含足够多的真实结构。强中心源常常能提供第一轮相位自校准所需的信噪比，但离轴弱源、扩展结构和谱结构若长期缺失，就会持续污染增益解。


![自校准的成像与求解闭环](figures/calibration_2gc_selfcal_loop.png)

**图 8.3.1** 自校准把成像、去卷积模型和增益求解耦合在一起。闭环可以放大正确模型带来的改进，也可以放大错误模型造成的偏差。


### 8.3.2 模型完整性与残差动态范围

一个最小化的一维实验足以说明自校准的双重性。设真实天空包含一个亮源和两个较弱离轴源。若初始模型只包含亮源，相位自校准会明显改善中心结构，因为强源提供了稳定的相位参考；但离轴源没有进入模型，求解器仍可能把它们的一部分结构误当成增益误差。只有当天空模型逐渐纳入这些离轴成分后，残差才会进一步下降。

这也是实践中自校准通常从相位自校准开始、再谨慎进入幅相自校准的原因。相位误差对图像动态范围影响很强，且相位自校准不直接改变通量标尺；幅相自校准则更有能力吸收模型误差，因此需要更可靠的天空模型和外部通量约束。


![模型完整性决定自校准能否继续降低残差](figures/calibration_2gc_model_completeness.png)

**图 8.3.2** 同一目标场在未自校准、不完整模型相位自校准、较完整模型幅相自校准三种情况下的一维脏图。自校准确实能改善图像，但模型越完整，残差才越接近真实噪声水平。


### 8.3.3 解间隔的权衡

自校准中的另一个核心选择是解间隔。解间隔越短，增益模型越能跟踪快速相位变化；但每个解使用的数据越少，信噪比越低，解也更容易被噪声支配。解间隔越长，热噪声会被更多数据平均掉；但真实增益在一个解间隔内若已经明显变化，求得的平均解就无法正确描述每个时刻。

因此，解间隔不是越短越好，也不是越长越稳定。合理的选择通常位于两类误差之间的折中点：热噪声误差随积分增加而下降，未跟踪的增益变化随间隔增加而上升。目标场强度、阵列灵敏度、天气稳定性和频率都会改变这个折中点。


![自校准解间隔的信噪比与跟踪能力折中](figures/calibration_2gc_solution_interval_tradeoff.png)

**图 8.3.3** 解间隔的概念性误差分解。短间隔受热噪声限制，长间隔受未跟踪增益变化限制，总残差通常存在一个较优区间。


### 8.3.4 典型失败模式

自校准失败常常不是因为求解器没有收敛，而是因为收敛到了错误问题的答案。初始模型过于贫乏时，扩展结构和弱源可能被增益吸收；总信噪比不足时，过短解间隔会产生噪声主导的解；幅相自校准过早时，通量标尺可能漂移；方向依赖残差若被方向无关增益强行拟合，则图像不同位置会出现不同类型的伪影。

稳健的 2GC 往往遵循保守顺序：先完成可靠的 1GC，检查数据和闭合量；再使用强结构做相位自校准；确认模型和残差稳定后，再考虑幅相自校准；若残差明显随方向变化，则停止把问题塞进方向无关增益，转向 3GC。


### 8.3.5 与 3GC 的连接

2GC 的基本近似是每台天线在整个目标场中只有一个方向无关复增益。这在小视场、方向依赖效应弱、目标场主导结构集中的情况下非常有效。但宽场低频观测、高动态范围连续谱成像和精密极化观测常常违反这一近似。主波束、指向、电离层和离轴极化泄漏会让不同天空方向需要不同 Jones 项。

一旦残差表现出位置相关性，继续缩短 2GC 解间隔或提高求解次数通常不能根治问题。下一节的 3GC 正是把这种方向依赖结构显式放入测量方程，使校准从“每台天线一个增益”扩展到“每台天线、每个方向、每段频率都有可建模响应”。
